# Yamazaki Shishi — SDXL LoRA training

このNotebookは **データ確認 → SDXL読み込み → LoRA適用 → 300 step学習 → LoRA保存** を一続きで実行します。

- `pipe` を使う前に同じ学習セル内で初期化します（`NameError: pipe is not defined` 対策）。
- LoRAを複数回追加する古い試行セルや、学習ループのないプレースホルダーは削除しました。
- この学習コードに不要な追加パッケージのインストールを外し、PyTorch環境の不用意な変更を避けます。
- Kaggle Notebook の Settings で **GPU と Internet を有効**にしてから実行してください。
- 出力先: `/kaggle/working/yamazaki_shishi_lora`

学習完了後、出力フォルダの `.safetensors` を Kaggle の Output から保存できます。

In [ ]:
import sys
import torch
import peft
import diffusers
from pathlib import Path

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Diffusers:", diffusers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), (
    "GPUが無効です。Kaggle Notebook Settings で Accelerator = GPU にしてください。"
)

# Kaggleのmount先は環境により異なるため、候補と再帰検索で画像を検出。
input_root = Path("/kaggle/input")
preferred = [
    input_root / "datasets/tyokouchi/yamazaki-shishi-sdxl-dataset/images",
    input_root / "yamazaki-shishi-sdxl-dataset/images",
]
image_exts = {".png", ".jpg", ".jpeg", ".webp"}
image_paths = []
for candidate in preferred:
    if candidate.is_dir():
        image_paths = sorted(
            p for p in candidate.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        )
        if image_paths:
            break
if not image_paths:
    image_paths = sorted(
        p for p in input_root.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
        and "yamazaki-shishi-sdxl-dataset" in str(p).lower()
    )
assert image_paths, (
    "学習画像が見つかりません。KaggleのAdd Inputで "
    "'yamazaki-shishi-sdxl-dataset' を追加してください。"
)
print("Images:", len(image_paths))
print("Sample:", image_paths[0])


In [ ]:
import gc
import random
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import StableDiffusionXLPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

OUTPUT_DIR = Path("/kaggle/working/yamazaki_shishi_lora")
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
TRIGGER = "yamazaki_shishi"
WIDTH, HEIGHT = 512, 288
TRAIN_STEPS = 300
LEARNING_RATE = 1e-5
SEED = 42
CAPTION = (
    f"{TRIGGER}, Japanese traditional shishimai lion dance, "
    "traditional Japanese lion costume"
)

# Reset a previous pipeline so the adapter cannot accidentally be added twice.
if "pipe" in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

class ShishiDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB").resize(
                (WIDTH, HEIGHT), Image.Resampling.LANCZOS
            )
            array = np.asarray(image, dtype=np.float32) / 127.5 - 1.0
        return {"pixel_values": torch.from_numpy(array).permute(2, 0, 1)}

loader = DataLoader(
    ShishiDataset(image_paths), batch_size=1, shuffle=True, num_workers=0
)
print("Loading SDXL base model:", MODEL_ID)
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, use_safetensors=True
).to("cuda")
scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# Freeze base weights. Keep VAE in FP32 for encoding stability.
pipe.vae.requires_grad_(False)
pipe.vae.to(dtype=torch.float32)
pipe.text_encoder.requires_grad_(False)
pipe.text_encoder_2.requires_grad_(False)
pipe.unet.requires_grad_(False)

lora_config = LoraConfig(
    r=4, lora_alpha=4, lora_dropout=0.0,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)
pipe.unet = get_peft_model(pipe.unet, lora_config)
trainable = [p for p in pipe.unet.parameters() if p.requires_grad]
for parameter in trainable:
    parameter.data = parameter.data.float()
assert trainable, "LoRA parameters were not created"
pipe.unet.print_trainable_parameters()
pipe.unet.enable_gradient_checkpointing()
pipe.unet.train()
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)

with torch.no_grad():
    encoded = pipe.encode_prompt(
        prompt=CAPTION, device="cuda",
        num_images_per_prompt=1, do_classifier_free_guidance=False,
    )
    prompt_embeds = encoded[0].detach().to(torch.float16)
    pooled_embeds = encoded[2].detach().to(torch.float16)
time_ids = torch.tensor(
    [[HEIGHT, WIDTH, 0, 0, HEIGHT, WIDTH]],
    device="cuda", dtype=torch.float16,
)

step, running_loss = 0, 0.0
while step < TRAIN_STEPS:
    for batch in loader:
        if step >= TRAIN_STEPS:
            break
        pixels = batch["pixel_values"].to("cuda", dtype=torch.float32)
        with torch.no_grad():
            latents = pipe.vae.encode(pixels).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
        if not torch.isfinite(latents).all():
            raise RuntimeError(f"NaN/Inf in VAE latents at step {step + 1}")
        latents = latents.to(torch.float16)
        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0, scheduler.config.num_train_timesteps,
            (latents.shape[0],), device="cuda"
        ).long()
        noisy_latents = scheduler.add_noise(latents, noise, timesteps)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            prediction = pipe.unet(
                noisy_latents, timesteps,
                encoder_hidden_states=prompt_embeds,
                added_cond_kwargs={
                    "text_embeds": pooled_embeds, "time_ids": time_ids,
                },
            ).sample
            target = noise
            if scheduler.config.prediction_type == "v_prediction":
                target = scheduler.get_velocity(latents, noise, timesteps)
            loss = F.mse_loss(prediction.float(), target.float(), reduction="mean")
        if not torch.isfinite(prediction).all():
            raise RuntimeError(f"NaN/Inf in UNet prediction at step {step + 1}")
        if not torch.isfinite(loss):
            raise RuntimeError(f"NaN/Inf loss at step {step + 1}: {loss.item()}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in trainable):
            raise RuntimeError(f"NaN/Inf gradient at step {step + 1}")
        optimizer.step()
        step += 1
        running_loss += loss.item()
        if step <= 5 or step % 25 == 0:
            print(
                f"Step {step}/{TRAIN_STEPS} | loss={loss.item():.6f} | "
                f"latents_finite={torch.isfinite(latents).all().item()} | "
                f"prediction_finite={torch.isfinite(prediction).all().item()}"
            )
        if step % 25 == 0:
            print(f"Mean loss (last 25): {running_loss / 25:.6f}")
            running_loss = 0.0

# Save the trained adapter only, not the full base model.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
state = get_peft_model_state_dict(pipe.unet)
state = convert_state_dict_to_diffusers(state)
StableDiffusionXLPipeline.save_lora_weights(
    save_directory=str(OUTPUT_DIR),
    unet_lora_layers=state,
    safe_serialization=True,
)
saved_files = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())
assert any(name.endswith(".safetensors") for name in saved_files), (
    "学習完了後に .safetensors が見つかりません。"
)
print("Training complete.")
print("LoRA output:", OUTPUT_DIR)
print("Saved files:", saved_files)
del optimizer, pipe
gc.collect()
torch.cuda.empty_cache()
